# Track B: Baseline Models, Evaluation and Prediction

Goal: build a Dummy baseline and a simple Linear Regression model, evaluate them
(cross-validation and test set), and make a prediction on a new listing.
Target: price_eur (London prices converted from GBP to EUR at 1.15).
Split: train_ids_eur.csv and test_ids_eur.csv provided by the team.

## 1. Load data

In [1]:
import numpy as np
import pandas as pd

GBP_TO_EUR = 1.15

df = pd.read_csv("hostwise_5cities.csv", low_memory=False)
df["price_num"] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce"
)
df["price_eur"] = df["price_num"] * np.where(df["city"] == "london", GBP_TO_EUR, 1.0)
print(df.shape)
print(df["city"].unique())

(245402, 93)
<StringArray>
['paris', 'madrid', 'barcelona', 'london', 'rome']
Length: 5, dtype: str


## 2. Train/test split

Rows are assigned by listing id using the team's train_ids_eur.csv and test_ids_eur.csv.
Rows with missing values in the features or the target are dropped (basic handling for Track B).
Features: 5 numerical (accommodates, bedrooms, bathrooms, latitude, longitude) and
3 categorical (city, room_type, property_type). Target: price_eur.

In [2]:
train_ids = pd.read_csv("train_ids_eur.csv")["id"]
test_ids = pd.read_csv("test_ids_eur.csv")["id"]

num_features = ["accommodates", "bedrooms", "bathrooms", "latitude", "longitude"]
cat_features = ["city", "room_type", "property_type"]
features = num_features + cat_features
target = "price_eur"

# Drop rows with missing values (Track B instruction)
train = df[df["id"].isin(train_ids)].dropna(subset=features + [target])
test = df[df["id"].isin(test_ids)].dropna(subset=features + [target])

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(X_train.shape, X_test.shape, y_train.isna().sum())

(95097, 8) (23797, 8) 0


In [3]:
print(y_train.describe())
print(y_test.describe())
print(y_test.sort_values().tail(10))

count    95097.000000
mean       324.103408
std        660.688028
min          7.030000
25%        147.030000
50%        222.530000
75%        350.152000
max      97003.050000
Name: price_eur, dtype: float64
count    23797.000000
mean       322.758047
std        601.791515
min         17.960000
25%        146.250000
50%        221.000000
75%        347.587500
max      57064.000000
Name: price_eur, dtype: float64
55901      9200.00
174719    10103.00
78108     10270.75
194439    10350.00
243551    10350.00
138295    13728.50
29855     13800.00
194414    15000.00
118969    15640.00
223672    57064.00
Name: price_eur, dtype: float64


## 3. Baseline models (test evaluation)

DummyRegressor (predicts the median price) vs LinearRegression with imputation and
one-hot encoding of the categorical features. Metrics: RMSE, MAE and R2 (in EUR).

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate(y_true, y_pred):
    return (np.sqrt(mean_squared_error(y_true, y_pred)),
            mean_absolute_error(y_true, y_pred),
            r2_score(y_true, y_pred))

def make_linear():
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features),
    ])
    return Pipeline([("prep", prep), ("model", LinearRegression())])

dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
lin = make_linear().fit(X_train, y_train)

rmse_d, mae_d, r2_d = evaluate(y_test, dummy.predict(X_test))
rmse_l, mae_l, r2_l = evaluate(y_test, lin.predict(X_test))

print(f"Dummy  -> RMSE: {rmse_d:.2f} | MAE: {mae_d:.2f} | R2: {r2_d:.3f}")
print(f"Linear -> RMSE: {rmse_l:.2f} | MAE: {mae_l:.2f} | R2: {r2_l:.3f}")

Dummy  -> RMSE: 610.07 | MAE: 178.57 | R2: -0.028
Linear -> RMSE: 568.24 | MAE: 159.87 | R2: 0.108


## 4. Cross-validation

5-fold cross-validation on the training set only (no test data used), comparing the
Dummy baseline and Linear Regression using RMSE and MAE (in EUR).

In [5]:
from sklearn.model_selection import cross_validate

scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}

cv_d = cross_validate(DummyRegressor(strategy="median"), X_train, y_train, cv=5, scoring=scoring)
cv_l = cross_validate(make_linear(), X_train, y_train, cv=5, scoring=scoring)

for name, cv in [("Dummy ", cv_d), ("Linear", cv_l)]:
    rmse, mae = -cv["test_rmse"], -cv["test_mae"]
    print(f"CV {name} RMSE: {rmse.mean():.2f} (+/- {rmse.std():.2f}) | "
          f"MAE: {mae.mean():.2f} (+/- {mae.std():.2f})")

CV Dummy  RMSE: 651.65 (+/- 148.93) | MAE: 178.90 (+/- 4.66)
CV Linear RMSE: 606.33 (+/- 157.39) | MAE: 160.40 (+/- 3.72)


## 5. Prediction on a new listing

Predicted nightly price (EUR) for a new listing (4 guests, 2 bedrooms, 1 bathroom, London coordinates).

In [6]:
new_listing = pd.DataFrame([{
    "accommodates": 4, "bedrooms": 2, "bathrooms": 1,
    "latitude": 51.5, "longitude": -0.12,
    "city": "london", "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
}])

pred = lin.predict(new_listing[features])[0]
print(f"Predicted nightly price (EUR): {pred:.2f}")

Predicted nightly price (EUR): 309.38


## Conclusion

Is our model better than a simple prediction?
Yes, moderately. Prices are in EUR (London converted from GBP at 1.15).
Rows with missing values were dropped, as instructed for Track B.

Cross-validation (5-fold, training set only): Linear Regression reaches MAE 160.40
vs 178.90 for the Dummy (median), and RMSE 606.33 vs 651.65.
Test set: MAE 159.87 vs 178.57 (about 10% lower error), RMSE 568.24 vs 610.07,
R2 0.108 vs -0.028.

The Linear model beats the Dummy consistently in cross-validation and on the test set,
but it explains only about 11% of the price variance. The features are few and the
preprocessing is minimal, since this is a baseline.

Next step: integrate the Track A preprocessing pipeline (Part 3) and compare against
this baseline.